In [1]:
# =====================================================================
# CELLA IMPORTAZIONI LIBRERIE PER MAC
# =====================================================================
import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log inutili del Mac
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Blocca tutto tranne gli errori fatali
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# NOTA: Rimosso 'TF_CUDNN_USE_AUTOTUNE' perché sul tuo Mac non serve

import tensorflow as tf

# Altri silenziatori di log 
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# =====================================================================
# MODIFICA 1: DISABILITARE LA GPU PER EVITARE I GRADIENTI NaN (ESPLOSIONE DELLA LOSS)
# =====================================================================
# Diciamo a TensorFlow di "nascondere" la GPU M1 (gestita da tensorflow-metal).
tf.config.set_visible_devices([], 'GPU')

# Riga di controllo per essere sicuri al 100% che abbia funzionato
print("Dispositivi di calcolo attivi:", tf.config.get_visible_devices())
# =====================================================================

# Import di Keras (lasciati identici a quelli del tuo collega)
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

Dispositivi di calcolo attivi: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
# ==============================================================================
# DATA ENGINE V9 (EMA prima, Transpose + Reshape poi)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        # 1. Calcolo magnitudo. Shape: (T, 6, 3, 120)
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        
        # 2. Inizializzazione EMA sul tensore originario
        bg = np.copy(mag[0])
        decluttered = np.zeros_like(mag)
        
        # 3. Calcolo EMA (opera in modo indipendente su ogni sensore e bin)
        for t in range(T):
            bg = alpha * mag[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag[t] - bg)
            
        # 4. TRASPOSIZIONE FONDAMENTALE: Spostiamo i 120 bin (asse 3) prima dei radar (asse 1)
        # Shape da (T, 6, 3, 120) diventa (T, 120, 6, 3)
        decluttered_transposed = np.transpose(decluttered, (0, 3, 1, 2))
        
        # 5. RESHAPE FINALE: Uniamo i radar(6) e le antenne(3) nei 18 canali, mantenendo intatti i bin
        # Shape finale: (T, 1, 120, 18)
        decluttered_final = decluttered_transposed.reshape(T, 1, 120, 18)
        
        # Flatten delle coordinate e concatenazione con la mask
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered_final) # <--- Ora aggiungiamo il tensore corretto
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ==============================================================================
# 2. SPLIT STRATIFICATO RIGOROSO
# ==============================================================================
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")

In [ ]:
# ==============================================================================
# DATA ENGINE V9 (EMA prima, Transpose + Reshape poi, Sliding Window Ottimizzata con NumPy Stride Tricks)
# ==============================================================================
from numpy.lib.stride_tricks import sliding_window_view

def load_and_process_all_files(file_list, alpha=0.20, W=5):
    X_all, Y_all = [], []
    print(f"Inizio caricamento con Sliding Window (W={W}) ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        if T < W:
            continue
            
        # 1. Calcolo magnitudo. Shape: (T, 6, 3, 120)
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        
        # 2. Calcolo EMA
        bg = np.copy(mag[0])
        decluttered = np.zeros_like(mag)
        for t in range(T):
            bg = alpha * mag[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag[t] - bg)
            
        # 3. Transpose e Reshape: portiamo i dati a (T, 120, 18)
        decluttered_reshaped = np.transpose(decluttered, (0, 3, 1, 2)).reshape(T, 120, 18)
        
        # Flatten coordinate e maschere. Shape: (T, 12)
        flat_coords = people_xy.reshape(T, 8)
        combined_target_full = np.concatenate([flat_coords, people_mask], axis=1)

        # 4. CREAZIONE FINESTRE OTTIMIZZATA (Zero-Copy)
        # Genera un tensore di shape (T - W + 1, 120, 18, 5)
        windows = sliding_window_view(decluttered_reshaped, window_shape=W, axis=0)
        
        # Spostiamo l'ultimo asse (i 5 frame) in posizione 1 -> Diventa (T - W + 1, 5, 120, 18)
        windows = np.moveaxis(windows, -1, 1)
        
        # I target corrispondenti partono dal frame W-1 fino alla fine -> Shape (T - W + 1, 12)
        targets = combined_target_full[W - 1 : T]

        X_all.append(windows)
        Y_all.append(targets)
        
        print(f"File {i+1}/{len(file_list)} processato. Generate {windows.shape[0]} finestre.")

    # Ora X_all è una lista di grossi blocchi già formattati correttamente, concatenate funzionerà perfettamente!
    X = np.concatenate(X_all, axis=0).astype(np.float32) if X_all else np.array([])
    Y = np.concatenate(Y_all, axis=0).astype(np.float32) if Y_all else np.array([])
    return X, Y

# ==============================================================================
# 2. SPLIT STRATIFICATO RIGOROSO
# ==============================================================================
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento con Sliding Window (W=5) ed EMA Decluttering di 18 file...
File 1/18 processato. Generate 7496 finestre.
File 2/18 processato. Generate 7496 finestre.
File 3/18 processato. Generate 7496 finestre.
File 4/18 processato. Generate 7496 finestre.
File 5/18 processato. Generate 7496 finestre.
File 6/18 processato. Generate 7496 finestre.
File 7/18 processato. Generate 7496 finestre.
File 8/18 processato. Generate 7496 finestre.
File 9/18 processato. Generate 7496 finestre.
File 10/18 processato. Generate 7496 finestre.
File 11/18 processato. Generate 7496 finestre.
File 12/18 processato. Generate 7496 finestre.
File 13/18 processato. Generate 7496 finestre.
File 14/18 processato. Generate 7496 finestre.
File 15/18 processato. Generate 7496 finestre.
File 16/18 processato. Generate 7496 finestre.
File 17/18 processato. Generate 7496 finestre.
File 18/18 processato. Generate 7496 finestre.

--- PREPARAZIONE VALIDATION SET ---
Inizio carica

In [3]:
# =====================================================================
# BULGARIAN SQUAT PER MAC
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V1-Heavy (Aggiornata per kernel orizzontali)
# ==============================================================================
def build_eeai_model_v1_heavy(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- BLOCCO 1: Estrazione Base ---
    # Nota: il kernel è (1, 5) per scorrere solo lungo i bin spaziali
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) # Dimezza i bin: 120 -> 60
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # Dimensione: 60 -> 30
    
    # --- BLOCCO 2: Livello Intermedio ---
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x) # Dimensione: 30 -> 15
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    # x = layers.MaxPooling2D((1, 2), name="pool_4")(x) # Rimosso per evitare che l'asse diventi 0!
    
    # --- BLOCCO 3: Sostituiti Conv2D con SeparableConv2D per risparmiare memoria Flash ---
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_5")(x) # Dimensione: 15 -> 7
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="conv_3b")(x)
    # Rimosso MaxPooling qui
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="conv_3c")(x)
    
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    x = layers.Dense(256, activation='relu', name="features_deep2")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V1_Heavy")
    


# Inizializzazione del nuovo modello
model_heavy = build_eeai_model_v1_heavy()

# Compilazione 
model_heavy.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_heavy = ModelCheckpoint("eeai_best_model_romano_heavy.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---")
history_light = model_heavy.fit(
    X_train, Y_train,                # Usiamo i tensori in RAM, non il generatore!
    validation_data=(X_val, Y_val),  # Usiamo i tensori in RAM!
    batch_size=32,                   # LA MAGIA AVVIENE QUI: 32 frame alla volta
    shuffle=True,                    # Rimescolamento perfetto per evitare bias
    epochs=EPOCHS,
    callbacks=[checkpoint_heavy, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V1-Heavy (Aggiornata per kernel orizzontali, Sliding Window)
# ==============================================================================
def build_eeai_model_v1_heavy(n_radars=6, n_antennas=3, n_bins=120, W=5):
    input_channels = n_radars * n_antennas
    # L'input ora accetta finestre di W frame (5, 120, 18)
    inputs = layers.Input(shape=(W, n_bins, input_channels), name="radar_input")

    # --- BLOCCO 1: Estrazione Spazio-Temporale ---
    # Usiamo un kernel (3, 5): analizza 3 frame consecutivi e 5 bin di distanza contemporaneamente
    x = layers.Conv2D(32, (3, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) # Riduce solo la distanza: 120 -> 60
    x = layers.Conv2D(32, (3, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # Riduce solo la distanza: 60 -> 30
    
    # --- BLOCCO 2: Livelli Separabili Leggeri ---
    x = layers.SeparableConv2D(64, (3, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x) # Riduce solo la distanza: 30 -> 15
    x = layers.SeparableConv2D(64, (3, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    
    # --- BLOCCO 3: Alta Astrazione (Depthwise Separable per Flash Budget) ---
    x = layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu', name="conv_3a")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_5")(x) # Riduce la distanza: 15 -> 7
    x = layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu', name="conv_3b")(x)
    x = layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu', name="conv_3c")(x)
    
    # Il GlobalAveragePooling2D collassa sia l'asse del tempo (5) che della distanza (7) in un unico vettore
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    x = layers.Dense(256, activation='relu', name="features_deep2")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V1_Heavy")

# Inizializzazione del nuovo modello
model_heavy = build_eeai_model_v1_heavy()

# Compilazione 
model_heavy.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_heavy = ModelCheckpoint("eeai_best_model_romano_heavy.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---")
history_light = model_heavy.fit(
    X_train, Y_train,                # Usiamo i tensori in RAM, non il generatore!
    validation_data=(X_val, Y_val),  # Usiamo i tensori in RAM!
    batch_size=32,                   # LA MAGIA AVVIENE QUI: 32 frame alla volta
    shuffle=True,                    # Rimescolamento perfetto per evitare bias
    epochs=EPOCHS,
    callbacks=[checkpoint_heavy, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---
Epoch 1/50
4215/4217 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - hungarian_mask_acc: 0.7508 - hungarian_rmse_metres: 0.9595 - loss: 2.2864
Epoch 1: val_loss improved from None to 0.78700, saving model to eeai_best_model_romano_heavy.keras

Epoch 1: finished saving model to eeai_best_model_romano_heavy.keras
4217/4217 ━━━━━━━━━━━━━━━━━━━━ 145s 34ms/step - hungarian_mask_acc: 0.8440 - hungarian_rmse_metres: 0.7314 - loss: 1.3076 - val_hungarian_mask_acc: 0.9100 - val_hungarian_rmse_metres: 0.5269 - val_loss: 0.7870 - learning_rate: 0.0010
Epoch 2/50
4215/4217 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - hungarian_mask_acc: 0.9491 - hungarian_rmse_metres: 0.5074 - loss: 0.5585
Epoch 2: val_loss improved from 0.78700 to 0.71935, saving model to eeai_best_model_romano_heavy.keras

Epoch 2: finished saving model to eeai_best_model_romano_heavy.keras
4217/4217 ━━━━━━━━━━━━━━━━━━━━ 147s 35ms/step - hungarian_mask_acc: 0.9527 - hungarian_rms

In [5]:
# ==============================================================================
# MODEL SUMMARY PER EMBEDDED
# ==============================================================================

def embedded_summary(model, input_shape=(1, 120, 18)):
    
    # 2. Calcola i parametri statici (Flash)
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
   # 3. Calcola il picco di memoria dinamica (SRAM/Tensor Arena)
    max_layer_ram_kb = 0
    for layer in model.layers:
        # AGGIUNTO: Salta l'InputLayer o i layer senza output_shape per evitare l'AttributeError
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    # 4. Stampa il verdetto 
    print("============================================")
    print("   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite : < 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite : < 400 KB)")
    print(" Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)")
    print(" Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)")
    print("============================================\n")

embedded_summary(model_heavy)

   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   
 Memoria FLASH stimata : 490.92 KB  (Limite : < 800 KB)
 Memoria SRAM stimata  : ~8.44 KB (Limite : < 400 KB)
 Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)
 Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)



In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.1 (Fix Output 12) nuovo
# ==============================================================================

#file_target = "dataset/data/window_000011.npz"
file_target = "dataset/window_000014.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V8)...")
    # 1. Calcolo magnitudo originale
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
    
    # 2. Calcolo EMA
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)
        
    # 3. Transpose e Reshape finale
    decluttered = np.transpose(decluttered, (0, 3, 1, 2)).reshape(T, 1, 120, 18)

    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Caricamento del modello 
    model_heavy = load_model(
        "eeai_best_model_romano_heavy.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_heavy.predict(decluttered, verbose=0)
    
    # ==========================================
    # IL FIX E' QUI: Slicing corretto per la V8
    # ==========================================
    # preds ha dimensione [T, 12]. 
    # Prendiamo le prime 8 colonne per le coordinate, e le ultime 4 per le maschere.
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato
            ax.set_title(f"Radar V9 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.2 (Aggiornato per Sliding Window W=5)
# ==============================================================================

file_target = "dataset/window_000014.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri in corso (Sliding Window)...")
    # 1. Calcolo magnitudo originale
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
    
    # 2. Calcolo EMA
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)
        
    # 3. Transpose e Reshape intermedio (SENZA l'altezza fissa a 1 per permettere la finestra)
    # Shape passa da (T, 6, 3, 120) a (T, 120, 18)
    decluttered_reshaped = np.transpose(decluttered, (0, 3, 1, 2)).reshape(T, 120, 18)

    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Caricamento del modello 
    model_heavy = load_model(
        "eeai_best_model_romano_heavy.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            
            # --- COSTRUZIONE DINAMICA DELLA FINESTRA TEMPORALE (W=5) ---
            W = 5
            start_idx = max(0, frame_idx - W + 1)
            
            if frame_idx < W - 1:
                # Gestione caso limite: all'inizio replichiamo i frame disponibili per riempire la finestra
                pad_size = W - (frame_idx + 1)
                window = decluttered_reshaped[0 : frame_idx + 1]
                padding = np.repeat(decluttered_reshaped[0:1], pad_size, axis=0)
                input_window = np.concatenate([padding, window], axis=0)
            else:
                input_window = decluttered_reshaped[start_idx : frame_idx + 1]
                
            # Aggiunta asse del batch: da (5, 120, 18) a (1, 5, 120, 18)
            input_tensor = np.expand_dims(input_window, axis=0)
            
            # Esecuzione predizione per il singolo frame target
            pred = model_heavy.predict(input_tensor, verbose=0)[0]
            
            # Slicing: prime 8 per le coordinate, ultime 4 per le maschere
            p_coords = pred[:8].reshape(4, 2)
            p_mask = pred[8:]
            
            # --- DISEGNO ---
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Radar V9 (Sliding Window) | Frame: {frame_idx}/{T-1} | Window: 14", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                # Disegno Ground Truth
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                # Disegno Predizione
                conf = float(p_mask[i])
                if conf >= soglia:
                    px, py = p_coords[i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    # Lo slider parte da W-1 (cioè 4) per evitare predizioni spurie all'inizio
    slider_frame = widgets.IntSlider(value=500, min=4, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri in corso (Sliding Window)...
Caricamento dei pesi migliori dal file .keras ...
Dati pronti! Inizializzazione Radar...
